In [2]:
from ensemble_util import run_combo, project_root
import pandas as pd
ROOT_DIR = project_root()

# print(f"Project root directory: {ROOT_DIR}")


In [3]:
df_criminal_law = pd.read_csv(
    ROOT_DIR / "data/model_grades/criminal_law/criminal_law_ground_truth_and_model_output.csv"
)


df_public_law = pd.read_csv(
    ROOT_DIR / "data/model_grades/public_law/public_law_ground_truth_and_model_output.csv"
)

In [4]:
# ----------------------------
# Model sets
# ----------------------------
model_large = ['mistral-large-2512', 'deepseek-v3.1-terminus', 'qwen3-next-80b-a3b-thinking']
model_small = ['qwen3-next-80b-a3b-thinking', 'gpt-oss-20b', 'qwen3-30b-a3b-thinking-2507']

MODEL_SETS = {
    "large": model_large,
    "small": model_small,
}

# ----------------------------
# Prefixes (ALL variants)
# ----------------------------
PREFIXES = {
    "criminal_law": [
        "prompt_v5_ts_ext_rubric_model_ha",
        "prompt_v4_ts_ext_model_ha",
        "prompt_v3_ts_ext_rubric_ha",
        "prompt_v1_ta_min_ha",
    ],
    "public_law": [
        "prompt_v5_ts_ext_rubric_model_dp",
        "prompt_v4_ts_ext_model_dp",
        "prompt_v3_ts_ext_rubric_dp",
        "prompt_v1_ta_min_dp",
    ],
}

# ----------------------------
# Datasets
# ----------------------------
DATASETS = {
    "criminal_law": df_criminal_law,
    "public_law": df_public_law,
}

# ----------------------------
# Gold columns per law area
# ----------------------------
GOLD_COLS = {
    "criminal_law": "points",
    "public_law": "MH_grade_mRPS",
}

# ----------------------------
# Common settings
# ----------------------------
suffix = "__extracted_element"
iters = (1, 2, 3)
aggs = ["median", "min"]

print("Starting ensemble grading analysis...")
print(f"Model sets:\nOpen-L: {MODEL_SETS['large']},\nOpen-S: {MODEL_SETS['small']}")

all_rows = []
i = 0
for law_area, df in DATASETS.items():
    gold_col = GOLD_COLS[law_area] 

    for model_set_name, models in MODEL_SETS.items():
        for prefix in PREFIXES[law_area]:

            header = f"{law_area} | gold={gold_col} | ensemble_model_set={model_set_name} | prefix={prefix}"
            print("\n" + "=" * len(header))
            print(header)
            print("=" * len(header))

            try:
                rows = run_combo(
                    df,
                    models,
                    prefix,
                    suffix=suffix,
                    iters=iters,
                    aggs=aggs,
                    gold_col=gold_col,
                )
            except Exception as e:
                print(f"SKIP (error): {type(e).__name__}: {e}")
                continue


            for agg in aggs:
                print(f"\n=== agg = {agg} ===")
                agg_rows = [x for x in rows if x["agg"] == agg]

                if not agg_rows:
                    print("(no qwk metrics found)")
                    continue

                for r in agg_rows:
                    print(f"{r['metric']:<50} {r['value']:>8.3f}")

            for r in rows:
                all_rows.append(
                    {
                        "law_area": law_area,
                        "gold_col": gold_col,
                        "ensemble_model_set": model_set_name,
                        "prefix": prefix,
                        **r,
                    }
                )
            i += 1
            print(f"\nCOMPLETED {i} combinations so far.")

Starting ensemble grading analysis...
Model sets:
Open-L: ['mistral-large-2512', 'deepseek-v3.1-terminus', 'qwen3-next-80b-a3b-thinking'],
Open-S: ['qwen3-next-80b-a3b-thinking', 'gpt-oss-20b', 'qwen3-30b-a3b-thinking-2507']

criminal_law | gold=points | ensemble_model_set=large | prefix=prompt_v5_ts_ext_rubric_model_ha

=== agg = median ===
point_mean_qwk_fisher                                 0.439
point_sd_qwk_across_runs_kappa_space                  0.024

=== agg = min ===
point_mean_qwk_fisher                                 0.402
point_sd_qwk_across_runs_kappa_space                  0.025

COMPLETED 1 combinations so far.

criminal_law | gold=points | ensemble_model_set=large | prefix=prompt_v4_ts_ext_model_ha

=== agg = median ===
point_mean_qwk_fisher                                 0.281
point_sd_qwk_across_runs_kappa_space                  0.020

=== agg = min ===
point_mean_qwk_fisher                                 0.368
point_sd_qwk_across_runs_kappa_space                